# Season-Stratified RH Threshold Sweep (Full Diagnostic)

One full 3-row diagnostic figure **per season** (Dry, Belg, Kiremt), each showing all 3 method pairings:

| Row | Content |
|-----|---------|
| **1** | Regression slope vs RH threshold (ideal = 1 highlighted) + R² |
| **2** | \|slope − 1\| distance from ideal |
| **3** | Stacked sample counts + low-RH fraction |

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / 'pyproject.toml').exists()), cwd)
notebook_dir = str((repo_root / 'notebooks').resolve())
data_root = Path(
    os.environ.get('AETHMODULAR_DATA_ROOT', str(repo_root / 'research' / 'ftir_hips_chem'))
).expanduser().resolve()

scripts_path = str((repo_root / 'research' / 'ftir_hips_chem' / 'scripts').resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from config import SITES, MAC_VALUE
from data_matching import (
    load_aethalometer_data, load_filter_data, add_base_filter_id,
    match_all_parameters, load_etad_factors_with_filter_ids,
)

print(f'Repo root: {repo_root}')

In [ ]:
# === Configuration ===

SEASONS = {
    'Dry Season': [10, 11, 12, 1, 2],
    'Belg Rainy Season': [3, 4, 5],
    'Kiremt Rainy Season': [6, 7, 8, 9],
}
SEASONS_ORDER = ['Dry Season', 'Belg Rainy Season', 'Kiremt Rainy Season']
SEASON_COLORS = {
    'Dry Season': '#E67E22',
    'Belg Rainy Season': '#27AE60',
    'Kiremt Rainy Season': '#3498DB',
}

def get_season_3(month):
    for season, months in SEASONS.items():
        if month in months:
            return season
    return None

FACTOR_TO_FRAC = {
    'GF3 (Charcoal)': 'charcoal_frac', 'GF2 (Wood Burning)': 'wood_frac',
    'GF5 (Fossil Fuel Combustion)': 'fossil_fuel_frac',
    'GF4 (Polluted Marine)': 'polluted_marine_frac',
    'GF1 (Sea Salt Mixed)': 'sea_salt_frac',
}
FACTOR_TO_CONC = {
    'K_F3 Charcoal (ug/m3)': 'charcoal_conc',
    'K_F2 Wood Burning (ug/m3)': 'wood_conc',
    'K_F5 Fossil Fuel Combustion (ug/m3)': 'fossil_fuel_conc',
    'K_F4 Polluted Marine (ug/m3)': 'polluted_marine_conc',
    'K_F1 Sea Salt Mixed (ug/m3)': 'sea_salt_conc',
}
frac_cols = list(FACTOR_TO_FRAC.values())
conc_cols = list(FACTOR_TO_CONC.values())

PAIRS_FOR_RH = [
    ('ftir_ec', 'hips_fabs', 'FTIR EC (\u00b5g/m\u00b3)', 'HIPS Fabs/MAC (\u00b5g/m\u00b3)'),
    ('ftir_ec', 'ir_bcc',    'FTIR EC (\u00b5g/m\u00b3)', 'Aeth IR BCc (\u00b5g/m\u00b3)'),
    ('hips_fabs', 'ir_bcc',  'HIPS Fabs/MAC (\u00b5g/m\u00b3)', 'Aeth IR BCc (\u00b5g/m\u00b3)'),
]

FONT = {'title': 16, 'axis': 14, 'tick': 12, 'legend': 11, 'annot': 11}
plt.rcParams.update({
    'font.size': FONT['tick'], 'axes.titlesize': FONT['title'],
    'axes.labelsize': FONT['axis'], 'legend.fontsize': FONT['legend'],
})

output_root = repo_root / 'artifacts' / 'notebook_outputs' / 'rh_sweep_seasonal'
plots_dir = output_root / 'plots'
plots_dir.mkdir(parents=True, exist_ok=True)
print(f'Outputs: {output_root}')

In [ ]:
# === Load & merge data ===

def resolve_met_path(*filenames):
    for base in [data_root / 'Weather Data' / 'Meteostat', Path(notebook_dir)]:
        for name in filenames:
            if (base / name).exists():
                return base / name
    raise FileNotFoundError(filenames)

met_daily = pd.read_csv(resolve_met_path(
    'Addis_Ababa_daily_Average_met_Data.csv',
    'Addis Ababa daily Average met Data.csv'), encoding='utf-8-sig')
met_daily['date'] = pd.to_datetime(met_daily['date'])
met_daily['date_only'] = met_daily['date'].dt.normalize()

met_hourly = pd.read_csv(resolve_met_path(
    'master_meteostat_AddisAbaba_63450_2022-12-01_2024-10-01.csv'))
met_hourly = met_hourly.replace('NA', np.nan)
met_hourly['time'] = pd.to_datetime(met_hourly['time'])
for col in ['temp', 'dwpt', 'rhum', 'prcp', 'wdir', 'wspd', 'pres']:
    met_hourly[col] = pd.to_numeric(met_hourly[col], errors='coerce')
met_hourly['date_only'] = met_hourly['time'].dt.normalize()

hourly_daily_agg = met_hourly.groupby('date_only').agg(
    rhum_mean=('rhum', 'mean'), wspd_mean=('wspd', 'mean'),
    wdir_mean=('wdir', 'mean'),
).reset_index()

# Source apportionment + BC/EC
factors_df = load_etad_factors_with_filter_ids()
factors_df = factors_df.rename(columns={**FACTOR_TO_FRAC, **FACTOR_TO_CONC})
aethalometer_data = load_aethalometer_data()
filter_data = load_filter_data()
filter_data = add_base_filter_id(filter_data)
df_aeth = aethalometer_data.get('Addis_Ababa')
bc_df = match_all_parameters('Addis_Ababa', 'ETAD', df_aeth, filter_data)

etad_filters = filter_data[filter_data['Site'] == 'ETAD'][['SampleDate', 'FilterId']].drop_duplicates()
etad_filters = etad_filters.rename(columns={'SampleDate': 'date', 'FilterId': 'base_filter_id'})
bc_df['date'] = pd.to_datetime(bc_df['date'])
etad_filters['date'] = pd.to_datetime(etad_filters['date'])
bc_with_id = pd.merge(bc_df, etad_filters, on='date', how='left')

factor_merge_cols = ['base_filter_id'] + frac_cols + conc_cols
available_merge_cols = [c for c in factor_merge_cols if c in factors_df.columns]
df = pd.merge(bc_with_id, factors_df[available_merge_cols], on='base_filter_id', how='left')
df['Month'] = df['date'].dt.month
df['season'] = df['Month'].apply(get_season_3)
df['date_only'] = df['date'].dt.normalize()

met_merge = met_daily[['date_only', 'tavg', 'tmin', 'tmax', 'prcp']].copy()
df = pd.merge(df, met_merge, on='date_only', how='left', suffixes=('', '_met'))
df = pd.merge(df, hourly_daily_agg, on='date_only', how='left')

df_rh = df.dropna(subset=['rhum_mean']).copy()
print(f'Dataset: {len(df_rh)} samples with RH data')
for s in SEASONS_ORDER:
    n = (df_rh['season'] == s).sum()
    rh_range = df_rh.loc[df_rh['season'] == s, 'rhum_mean']
    print(f'  {s}: n={n}, RH range {rh_range.min():.0f}\u2013{rh_range.max():.0f}%')

---

## Per-Season RH Threshold Sweep (Full 3-Row Diagnostic)

One figure per season, each with:
- **Row 1**: Slopes + ideal line (green = 1.0) + R²
- **Row 2**: |slope − 1| distance from ideal
- **Row 3**: Stacked sample counts + low-RH fraction

---

In [ ]:
# =============================================================================
# Per-Season Full Diagnostic Sweep
# =============================================================================

rh_thresholds = np.arange(30, 90, 5)

# Store all results for the summary table
all_season_results = {}

for season in SEASONS_ORDER:
    season_df = df_rh[df_rh['season'] == season].copy()
    season_clr = SEASON_COLORS[season]

    fig, axes = plt.subplots(3, 3, figsize=(18, 14),
                              gridspec_kw={'height_ratios': [2, 1, 1]})

    season_results = {}

    for idx, (x_col, y_col, x_label, y_label) in enumerate(PAIRS_FOR_RH):
        ax_slope = axes[0, idx]
        ax_dist  = axes[1, idx]
        ax_n     = axes[2, idx]
        pair_key = f'{y_col}_vs_{x_col}'

        slopes_lo, slopes_hi = [], []
        r2_lo, r2_hi = [], []
        n_lo_list, n_hi_list = [], []

        for rh_thresh in rh_thresholds:
            lo = season_df[season_df['rhum_mean'] <= rh_thresh].dropna(subset=[x_col, y_col])
            hi = season_df[season_df['rhum_mean'] > rh_thresh].dropna(subset=[x_col, y_col])
            n_lo_list.append(len(lo))
            n_hi_list.append(len(hi))

            if len(lo) >= 5:
                s, _, r, _, _ = stats.linregress(lo[x_col], lo[y_col])
                slopes_lo.append(s); r2_lo.append(r**2)
            else:
                slopes_lo.append(np.nan); r2_lo.append(np.nan)

            if len(hi) >= 5:
                s, _, r, _, _ = stats.linregress(hi[x_col], hi[y_col])
                slopes_hi.append(s); r2_hi.append(r**2)
            else:
                slopes_hi.append(np.nan); r2_hi.append(np.nan)

        season_results[pair_key] = {
            'slopes_lo': slopes_lo, 'slopes_hi': slopes_hi,
            'r2_lo': r2_lo, 'r2_hi': r2_hi,
            'n_lo': n_lo_list, 'n_hi': n_hi_list,
        }

        n_lo_arr = np.array(n_lo_list)
        n_hi_arr = np.array(n_hi_list)
        n_tot    = n_lo_arr + n_hi_arr
        frac_lo  = np.where(n_tot > 0, n_lo_arr / n_tot * 100, 0)

        s_lo_arr = np.array(slopes_lo)
        s_hi_arr = np.array(slopes_hi)
        dist_lo  = np.abs(s_lo_arr - 1.0)
        dist_hi  = np.abs(s_hi_arr - 1.0)

        # ---- Row 1: slopes ----
        ax_slope.axhline(y=1.0, color='green', linewidth=2.5, alpha=0.6,
                         label='Ideal (slope = 1)')
        ax_slope.axhspan(0.9, 1.1, color='green', alpha=0.07)
        ax_slope.plot(rh_thresholds, slopes_lo, 'o-', color='#E67E22', linewidth=2,
                      markersize=6, label='Low-RH (RH $\leq$ threshold)')
        ax_slope.plot(rh_thresholds, slopes_hi, 's-', color='#3498DB', linewidth=2,
                      markersize=6, label='High-RH (RH > threshold)')

        ax2 = ax_slope.twinx()
        ax2.plot(rh_thresholds, r2_lo, 'o--', color='#E67E22', linewidth=1,
                 markersize=4, alpha=0.4)
        ax2.plot(rh_thresholds, r2_hi, 's--', color='#3498DB', linewidth=1,
                 markersize=4, alpha=0.4)
        ax2.set_ylabel('R$^2$ (dashed)', fontsize=FONT['axis'] - 2, alpha=0.6)
        ax2.set_ylim(0, 1)

        pair_title = f'{y_label.split("(")[0].strip()}\nvs {x_label.split("(")[0].strip()}'
        ax_slope.set_title(pair_title, fontsize=FONT['title'])
        ax_slope.set_ylabel('Regression Slope', fontsize=FONT['axis'])
        if idx == 0:
            ax_slope.legend(fontsize=FONT['legend'] - 1, loc='upper left')
        ax_slope.grid(True, alpha=0.3)
        ax_slope.set_xticklabels([])

        # ---- Row 2: |slope - 1| ----
        ax_dist.fill_between(rh_thresholds, dist_lo, alpha=0.3, color='#E67E22')
        ax_dist.fill_between(rh_thresholds, dist_hi, alpha=0.3, color='#3498DB')
        ax_dist.plot(rh_thresholds, dist_lo, 'o-', color='#E67E22', linewidth=2,
                     markersize=5, label='Low-RH')
        ax_dist.plot(rh_thresholds, dist_hi, 's-', color='#3498DB', linewidth=2,
                     markersize=5, label='High-RH')
        ax_dist.axhline(y=0, color='green', linewidth=1.5, alpha=0.5)
        ax_dist.set_ylabel('|slope $-$ 1|', fontsize=FONT['axis'] - 1)
        if idx == 2:
            ax_dist.legend(fontsize=FONT['legend'] - 2, loc='upper right', ncol=2)
        ax_dist.grid(True, alpha=0.3)
        ax_dist.set_xticklabels([])

        # Annotate best low-RH threshold
        valid_lo = ~np.isnan(dist_lo) & (n_lo_arr >= 5)
        if valid_lo.any():
            best_idx = np.where(valid_lo, dist_lo, np.inf).argmin()
            ax_dist.annotate(
                f'Best: {rh_thresholds[best_idx]}%\n'
                f'|s-1|={dist_lo[best_idx]:.3f}\nn={n_lo_arr[best_idx]}',
                xy=(rh_thresholds[best_idx], dist_lo[best_idx]),
                xytext=(rh_thresholds[best_idx] + 8, max(dist_lo[best_idx], 0.02) + 0.04),
                fontsize=8, color='#E67E22', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#E67E22', lw=1.5))

        # Annotate best high-RH threshold
        valid_hi = ~np.isnan(dist_hi) & (n_hi_arr >= 5)
        if valid_hi.any():
            best_idx_hi = np.where(valid_hi, dist_hi, np.inf).argmin()
            ax_dist.annotate(
                f'Best: {rh_thresholds[best_idx_hi]}%\n'
                f'|s-1|={dist_hi[best_idx_hi]:.3f}\nn={n_hi_arr[best_idx_hi]}',
                xy=(rh_thresholds[best_idx_hi], dist_hi[best_idx_hi]),
                xytext=(rh_thresholds[best_idx_hi] - 12, max(dist_hi[best_idx_hi], 0.02) + 0.06),
                fontsize=8, color='#3498DB', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#3498DB', lw=1.5))

        # ---- Row 3: sample counts ----
        ax_n.bar(rh_thresholds, n_lo_arr, width=3.5, color='#E67E22',
                 alpha=0.7, label='n (low-RH)')
        ax_n.bar(rh_thresholds, n_hi_arr, width=3.5, bottom=n_lo_arr,
                 color='#3498DB', alpha=0.7, label='n (high-RH)')

        for i, rh_t in enumerate(rh_thresholds):
            if n_tot[i] > 0:
                ax_n.text(rh_t, n_tot[i] + 0.5,
                          f'{frac_lo[i]:.0f}%', ha='center', fontsize=8,
                          fontweight='bold', color='#E67E22')

        ax_n.set_xlabel('RH Threshold (%)', fontsize=FONT['axis'])
        ax_n.set_ylabel('Sample Count', fontsize=FONT['axis'] - 1)
        if idx == 0:
            ax_n.legend(fontsize=FONT['legend'] - 2, loc='upper left', ncol=2)
        ax_n.grid(True, axis='y', alpha=0.3)

        ax_r = ax_n.twinx()
        ax_r.plot(rh_thresholds, frac_lo, 'D-', color='#8E44AD', linewidth=1.5,
                  markersize=4, alpha=0.8)
        ax_r.set_ylabel('% low-RH', fontsize=FONT['axis'] - 2, color='#8E44AD')
        ax_r.set_ylim(0, 100)
        ax_r.tick_params(axis='y', labelcolor='#8E44AD')

    all_season_results[season] = season_results

    season_short = season.split()[0]
    plt.suptitle(
        f'{season}\n'
        f'RH Threshold Sweep: Slopes, Distance from Ideal (1:1), and Sample Split',
        fontsize=FONT['title'] + 1, fontweight='bold', y=1.03,
        color=season_clr)
    plt.tight_layout()
    plt.savefig(str(plots_dir / f'rh_sweep_full_{season_short.lower()}.png'),
                dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# =============================================================================
# Summary Table: Best Threshold per Season \u00d7 Pairing
# =============================================================================

print('=' * 110)
print('BEST RH THRESHOLD PER SEASON \u00d7 METHOD PAIRING')
print('(threshold where low-RH subset slope is closest to 1, with n \u2265 5)')
print('=' * 110)

rows = []

for season in SEASONS_ORDER:
    for x_col, y_col, x_label, y_label in PAIRS_FOR_RH:
        pair_key = f'{y_col}_vs_{x_col}'
        pair_name = f'{y_label.split("(")[0].strip()} vs {x_label.split("(")[0].strip()}'
        res = all_season_results[season][pair_key]

        s_lo = np.array(res['slopes_lo'])
        s_hi = np.array(res['slopes_hi'])
        n_lo = np.array(res['n_lo'])
        n_hi = np.array(res['n_hi'])
        r2_lo = np.array(res['r2_lo'])

        # Best threshold for low-RH subset
        valid = ~np.isnan(s_lo) & (n_lo >= 5)
        if valid.any():
            dist = np.where(valid, np.abs(s_lo - 1.0), np.inf)
            best_i = dist.argmin()
            rows.append({
                'Season': season, 'Pairing': pair_name,
                'Best RH (%)': rh_thresholds[best_i],
                'Slope (low)': s_lo[best_i],
                '|s-1|': dist[best_i],
                'R\u00b2 (low)': r2_lo[best_i],
                'n (low)': n_lo[best_i],
                'n (high)': n_hi[best_i],
            })
        else:
            rows.append({
                'Season': season, 'Pairing': pair_name,
                'Best RH (%)': np.nan, 'Slope (low)': np.nan,
                '|s-1|': np.nan, 'R\u00b2 (low)': np.nan,
                'n (low)': 0, 'n (high)': 0,
            })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False, float_format='%.3f'))

In [ ]:
# =============================================================================
# Cross-Season Comparison: overlay all seasons on one figure
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10),
                          gridspec_kw={'height_ratios': [2, 1]})

markers_season = {'Dry Season': 'o', 'Belg Rainy Season': 's', 'Kiremt Rainy Season': '^'}

for idx, (x_col, y_col, x_label, y_label) in enumerate(PAIRS_FOR_RH):
    ax_slope = axes[0, idx]
    ax_dist  = axes[1, idx]
    pair_key = f'{y_col}_vs_{x_col}'

    ax_slope.axhline(y=1.0, color='green', linewidth=2.5, alpha=0.5,
                     label='Ideal (slope = 1)')
    ax_slope.axhspan(0.9, 1.1, color='green', alpha=0.05)
    ax_dist.axhline(y=0, color='green', linewidth=1.5, alpha=0.5)

    for season in SEASONS_ORDER:
        res = all_season_results[season][pair_key]
        s_lo = np.array(res['slopes_lo'])
        n_lo = np.array(res['n_lo'])
        dist_lo = np.abs(s_lo - 1.0)
        clr = SEASON_COLORS[season]
        mkr = markers_season[season]
        lbl = season.split()[0]  # short label

        ax_slope.plot(rh_thresholds, s_lo, f'{mkr}-', color=clr, linewidth=2,
                      markersize=6, label=f'{lbl} (low-RH slope)')
        ax_dist.plot(rh_thresholds, dist_lo, f'{mkr}-', color=clr, linewidth=2,
                     markersize=5, label=lbl)

        # Annotate n at key thresholds
        for rh_ann in [40, 50, 60]:
            if rh_ann in list(rh_thresholds):
                i_a = list(rh_thresholds).index(rh_ann)
                if not np.isnan(s_lo[i_a]):
                    ax_slope.annotate(f'n={n_lo[i_a]}',
                        (rh_ann, s_lo[i_a]),
                        textcoords='offset points', xytext=(0, 8),
                        fontsize=7, color=clr, ha='center', alpha=0.8)

    pair_title = f'{y_label.split("(")[0].strip()} vs {x_label.split("(")[0].strip()}'
    ax_slope.set_title(pair_title, fontsize=FONT['title'])
    ax_slope.set_ylabel('Slope (low-RH subset)', fontsize=FONT['axis'])
    ax_slope.grid(True, alpha=0.3)
    ax_slope.set_xticklabels([])
    if idx == 0:
        ax_slope.legend(fontsize=FONT['legend'] - 1, loc='upper left')

    ax_dist.set_xlabel('RH Threshold (%)', fontsize=FONT['axis'])
    ax_dist.set_ylabel('|slope $-$ 1|', fontsize=FONT['axis'] - 1)
    ax_dist.grid(True, alpha=0.3)
    if idx == 2:
        ax_dist.legend(fontsize=FONT['legend'] - 1, loc='upper right')

plt.suptitle('Cross-Season Comparison: Low-RH Slope vs RH Threshold\n'
             '(How close does each season get to ideal slope = 1?)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(str(plots_dir / 'rh_sweep_cross_season_comparison.png'),
            dpi=200, bbox_inches='tight')
plt.show()